# TCGA-BRCA Clinical Core Field Audit Review

This notebook is review-only. It loads the latest saved clinical core field-audit outputs from disk, checks audit-level summaries, and writes review tables for human source audit.

Important reminders:

- this notebook remains part of source audit
- this notebook does not freeze the cohort
- this notebook does not freeze the endpoint
- this notebook does not parse XML or rebuild parsed clinical tables


## Load the latest saved clinical core field-audit run

This section confirms that the stable latest-pointer exists and points to a completed clinical core field-audit run.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root / '01-data' / 'audit' / 'tcga-brca' / 'variables' / 'tcga_brca_clinical_core_field_audit_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest clinical core field-audit pointer not found: {latest_pointer_path}. Run the audit script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
audit_tsv_path = repo_root / latest_pointer['clinical_core_field_audit_tsv']
group_summary_path = repo_root / latest_pointer['clinical_core_field_group_summary_tsv']
missingness_summary_path = repo_root / latest_pointer['clinical_core_missingness_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_pointer]))


,updated_at_utc,audit_run_id,parse_run_id,source_run_id,audit_run_directory,clinical_core_field_audit_tsv,clinical_core_field_group_summary_tsv,clinical_core_missingness_summary_tsv,run_log_json,clinical_biotab_latest_json,field_inventory_tsvs,core_table_count,field_count
0,2026-04-12T02:38:39Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,01-data/audit/tcga-brca/variables/clinical_cor...,01-data/audit/tcga-brca/variables/clinical_cor...,01-data/audit/tcga-brca/variables/clinical_cor...,01-data/audit/tcga-brca/variables/clinical_cor...,01-data/audit/tcga-brca/variables/clinical_cor...,01-data/audit/tcga-brca/variables/tcga_brca_cl...,{'clinical_patient': '01-data/audit/tcga-brca/...,4,171


## Load saved audit artifacts

This section reads the saved combined audit TSV, field-group summary TSV, missingness summary TSV, and run log from disk only.


In [2]:
audit_df = pd.read_csv(audit_tsv_path, sep='\t')
group_summary_df = pd.read_csv(group_summary_path, sep='\t')
missingness_df = pd.read_csv(missingness_summary_path, sep='\t')
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

print(f'Audit TSV: {audit_tsv_path}')
print(f'Group summary TSV: {group_summary_path}')
print(f'Missingness summary TSV: {missingness_summary_path}')
print(f'Run log: {run_log_path}')
print(f"Audit run ID: {latest_pointer['audit_run_id']}")
print(f"Parse run ID: {latest_pointer['parse_run_id']}")
print(f"Source run ID: {latest_pointer['source_run_id']}")
display(pd.DataFrame([run_log['validation']]))
display(audit_df.head(10))


Audit TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\clinical_core_field_audit_runs\20260412T023839Z\clinical_core_field_audit.tsv
Group summary TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\clinical_core_field_audit_runs\20260412T023839Z\clinical_core_field_group_summary.tsv
Missingness summary TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\clinical_core_field_audit_runs\20260412T023839Z\clinical_core_missingness_summary.tsv
Run log: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\clinical_core_field_audit_runs\20260412T023839Z\run_log.json
Audit run ID: 20260412T023839Z
Parse run ID: 20260412T010932Z
Source run ID: 20260412T000556Z


,passed,required_core_table_count,resolved_core_table_count,row_count_positive_for_all_tables,schema_row_counts_match_table_column_counts,field_inventory_row_counts_match_table_column_counts,combined_field_count,expected_combined_field_count,combined_field_count_matches_expected,no_prior_run_overwrite
0,True,4,4,True,True,True,171,171,True,True


,audit_run_id,parse_run_id,source_run_id,table_name,field_name,source_position,alternate_column_name,cde_id_raw,row_count,non_missing_count,...,not_available_count,not_applicable_count,unknown_count,not_evaluated_count,discrepancy_count,na_count,n_a_count,null_count,none_count,nan_count
0,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,bcr_patient_uuid,1,bcr_patient_uuid,CDE_ID:,1097,1097,...,0,0,0,0,0,0,0,0,0,0
1,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,bcr_patient_barcode,2,bcr_patient_barcode,CDE_ID:2003301,1097,1097,...,0,0,0,0,0,0,0,0,0,0
2,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,form_completion_date,3,form_completion_date,CDE_ID:,1097,1097,...,0,0,0,0,0,0,0,0,0,0
3,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,prospective_collection,4,tissue_prospective_collection_indicator,CDE_ID:3088492,1097,1093,...,4,0,0,0,0,0,0,0,0,0
4,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,retrospective_collection,5,tissue_retrospective_collection_indicator,CDE_ID:3088528,1097,1093,...,4,0,0,0,0,0,0,0,0,0
5,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,birth_days_to,6,days_to_birth,CDE_ID:3008233,1097,1082,...,15,0,0,0,0,0,0,0,0,0
6,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,gender,7,gender,CDE_ID:2200604,1097,1097,...,0,0,0,0,0,0,0,0,0,0
7,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,menopause_status,8,menopause_status,CDE_ID:2957270,1097,1007,...,67,0,17,6,0,0,0,0,0,0
8,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,race,9,race,CDE_ID:2192199,1097,1002,...,92,0,0,3,0,0,0,0,0,0
9,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,ethnicity,10,ethnicity,CDE_ID:2192217,1097,923,...,163,0,8,3,0,0,0,0,0,0


## Save field counts by table


In [3]:
field_counts_by_table_df = (
    audit_df.groupby('table_name', as_index=False)
    .agg(
        field_count=('field_name', 'count'),
        row_count=('row_count', 'max'),
        non_missing_any_field_count=('non_missing_count', lambda series: int((series > 0).sum())),
        high_missingness_field_count=('missing_like_fraction', lambda series: int((series >= 0.75).sum())),
    )
    .sort_values('table_name')
    .reset_index(drop=True)
)
field_counts_by_table_path = results_root / '18_clinical_core_field_counts_by_table.tsv'
field_counts_by_table_df.to_csv(field_counts_by_table_path, sep='\t', index=False)

print(f'Saved: {field_counts_by_table_path}')
display(field_counts_by_table_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\18_clinical_core_field_counts_by_table.tsv


,table_name,field_count,row_count,non_missing_any_field_count,high_missingness_field_count
0,clinical_drug,28,2406,23,8
1,clinical_follow_up_v4_0,13,716,13,1
2,clinical_patient,112,1097,92,62
3,clinical_radiation,18,618,17,2


## Save audited fields by table


In [4]:
fields_by_table_columns = [
    'table_name',
    'source_position',
    'field_name',
    'alternate_column_name',
    'cde_id_raw',
    'probable_field_group',
    'group_assignment_rule',
    'row_count',
    'non_missing_count',
    'missing_like_count',
    'missing_like_fraction',
    'distinct_non_missing_count',
    'example_values_small_sample',
]
fields_by_table_df = audit_df[fields_by_table_columns].sort_values(['table_name', 'source_position']).reset_index(drop=True)
fields_by_table_path = results_root / '19_clinical_core_fields_by_table.tsv'
fields_by_table_df.to_csv(fields_by_table_path, sep='\t', index=False)

print(f'Saved: {fields_by_table_path}')
display(fields_by_table_df.head(20))


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\19_clinical_core_fields_by_table.tsv


,table_name,source_position,field_name,alternate_column_name,cde_id_raw,probable_field_group,group_assignment_rule,row_count,non_missing_count,missing_like_count,missing_like_fraction,distinct_non_missing_count,example_values_small_sample
0,clinical_drug,1,bcr_patient_uuid,bcr_patient_uuid,CDE_ID:,identifier / admin,identifier_prefix:bcr_,2406,2406,0,0.000000,780,"[""C07B122E-AC50-4DB2-ADD2-5617A5D0E976"", ""9435..."
1,clinical_drug,2,bcr_patient_barcode,bcr_patient_barcode,CDE_ID:2003301,identifier / admin,identifier_prefix:bcr_,2406,2406,0,0.000000,780,"[""TCGA-GM-A2DA"", ""TCGA-AO-A03N"", ""TCGA-A2-A0EW..."
2,clinical_drug,3,bcr_drug_barcode,bcr_drug_barcode,CDE_ID:,identifier / admin,identifier_prefix:bcr_,2406,2406,0,0.000000,2406,"[""TCGA-3C-AAAU-D60350"", ""TCGA-3C-AALI-D62900"",..."
3,clinical_drug,4,bcr_drug_uuid,bcr_drug_uuid,CDE_ID:,identifier / admin,identifier_prefix:bcr_,2406,2406,0,0.000000,2406,"[""00300D73-B562-4EE9-A8D4-27E5B97AE501"", ""0051..."
4,clinical_drug,5,form_completion_date,form_completion_date,CDE_ID:,identifier / admin,identifier_exact_name:form_completion_date,2406,2406,0,0.000000,345,"[""2012-12-6"", ""2011-1-10"", ""2010-9-19"", ""2013-..."
5,clinical_drug,6,pharmaceutical_therapy_drug_name,drug_name,CDE_ID:2975232,treatment / drug,keyword:drug,2406,2390,16,0.006650,203,"[""Cytoxan"", ""Tamoxifen"", ""Cyclophosphamide"", ""..."
6,clinical_drug,7,clinical_trial_drug_classification,clinical_trail_drug_classification,CDE_ID:3378323,stage,keyword:clinical_t,2406,3,2403,0.998753,3,"[""Antimetabolite"", ""Biological Therapy/Monoclo..."
7,clinical_drug,8,pharmaceutical_therapy_type,therapy_type,CDE_ID:2793530,treatment / drug,keyword:pharm,2406,2399,7,0.002909,8,"[""Chemotherapy"", ""Hormone Therapy"", ""Immunothe..."
8,clinical_drug,9,pharmaceutical_tx_started_days_to,days_to_drug_therapy_start,CDE_ID:3392465,treatment / drug,keyword:pharm,2406,2288,118,0.049044,486,"[""50"", ""61"", ""31"", ""62"", ""63""]"
9,clinical_drug,10,pharmaceutical_tx_ongoing_indicator,therapy_ongoing,CDE_ID:3103479,treatment / drug,keyword:pharm,2406,2391,15,0.006234,2,"[""NO"", ""YES""]"


## Save field-group distribution by table


In [5]:
field_group_distribution_df = group_summary_df.sort_values(['table_name', 'probable_field_group']).reset_index(drop=True)
field_group_distribution_path = results_root / '20_clinical_core_field_group_distribution.tsv'
field_group_distribution_df.to_csv(field_group_distribution_path, sep='\t', index=False)

print(f'Saved: {field_group_distribution_path}')
display(field_group_distribution_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\20_clinical_core_field_group_distribution.tsv


,audit_run_id,parse_run_id,source_run_id,table_name,probable_field_group,field_count,field_fraction_of_table,non_missing_any_field_count,high_missingness_field_count,field_names_json
0,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,demographics,0,0.000000,0,0,[]
1,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,diagnosis / pathology,0,0.000000,0,0,[]
2,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,follow-up / outcome-like,1,0.035714,1,0,"[""treatment_best_response""]"
3,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,identifier / admin,5,0.178571,5,0,"[""bcr_patient_uuid"", ""bcr_patient_barcode"", ""b..."
4,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,other / unclear,0,0.000000,0,0,[]
5,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,radiation,0,0.000000,0,0,[]
6,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,receptor / biomarker,0,0.000000,0,0,[]
7,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,stage,2,0.071429,2,1,"[""clinical_trial_drug_classification"", ""tx_on_..."
8,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,treatment / drug,20,0.714286,15,7,"[""pharmaceutical_therapy_drug_name"", ""pharmace..."
9,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,demographics,0,0.000000,0,0,[]


## Save high-completeness and high-missingness review tables


In [6]:
high_completeness_df = (
    audit_df.loc[(audit_df['missing_like_fraction'] <= 0.10) & (audit_df['non_missing_count'] > 0)]
    .sort_values(['table_name', 'missing_like_fraction', 'non_missing_count', 'source_position'], ascending=[True, True, False, True])
    .reset_index(drop=True)
)
high_completeness_path = results_root / '21_clinical_core_high_completeness_fields.tsv'
high_completeness_df.to_csv(high_completeness_path, sep='\t', index=False)

print(f'Saved: {high_completeness_path}')
display(high_completeness_df.head(25))


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\21_clinical_core_high_completeness_fields.tsv


,audit_run_id,parse_run_id,source_run_id,table_name,field_name,source_position,alternate_column_name,cde_id_raw,row_count,non_missing_count,...,not_available_count,not_applicable_count,unknown_count,not_evaluated_count,discrepancy_count,na_count,n_a_count,null_count,none_count,nan_count
0,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,bcr_patient_uuid,1,bcr_patient_uuid,CDE_ID:,2406,2406,...,0,0,0,0,0,0,0,0,0,0
1,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,bcr_patient_barcode,2,bcr_patient_barcode,CDE_ID:2003301,2406,2406,...,0,0,0,0,0,0,0,0,0,0
2,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,bcr_drug_barcode,3,bcr_drug_barcode,CDE_ID:,2406,2406,...,0,0,0,0,0,0,0,0,0,0
3,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,bcr_drug_uuid,4,bcr_drug_uuid,CDE_ID:,2406,2406,...,0,0,0,0,0,0,0,0,0,0
4,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,form_completion_date,5,form_completion_date,CDE_ID:,2406,2406,...,0,0,0,0,0,0,0,0,0,0
5,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_therapy_type,8,therapy_type,CDE_ID:2793530,2406,2399,...,7,0,0,0,0,0,0,0,0,0
6,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_tx_ongoing_indicator,10,therapy_ongoing,CDE_ID:3103479,2406,2391,...,15,0,0,0,0,0,0,0,0,0
7,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_therapy_drug_name,6,drug_name,CDE_ID:2975232,2406,2390,...,15,0,1,0,0,0,0,0,0,0
8,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_tx_started_days_to,9,days_to_drug_therapy_start,CDE_ID:3392465,2406,2288,...,116,0,0,0,2,0,0,0,0,0
9,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,bcr_patient_uuid,1,bcr_patient_uuid,CDE_ID:,716,716,...,0,0,0,0,0,0,0,0,0,0


In [7]:
high_missingness_df = (
    audit_df.loc[audit_df['missing_like_fraction'] >= 0.75]
    .sort_values(['table_name', 'missing_like_fraction', 'source_position'], ascending=[True, False, True])
    .reset_index(drop=True)
)
high_missingness_path = results_root / '22_clinical_core_high_missingness_fields.tsv'
high_missingness_df.to_csv(high_missingness_path, sep='\t', index=False)

print(f'Saved: {high_missingness_path}')
display(high_missingness_df.head(25))


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\22_clinical_core_high_missingness_fields.tsv


,audit_run_id,parse_run_id,source_run_id,table_name,field_name,source_position,alternate_column_name,cde_id_raw,row_count,non_missing_count,...,not_available_count,not_applicable_count,unknown_count,not_evaluated_count,discrepancy_count,na_count,n_a_count,null_count,none_count,nan_count
0,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,days_to_stem_cell_transplantation,13,days_to_stem_cell_transplantation,CDE_ID:3414613,2406,0,...,2406,0,0,0,0,0,0,0,0,0
1,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharm_regimen,14,pharm_regimen,CDE_ID:3366758,2406,0,...,2406,0,0,0,0,0,0,0,0,0
2,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharm_regimen_other,15,pharm_regimen_other,CDE_ID:3366930,2406,0,...,2406,0,0,0,0,0,0,0,0,0
3,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,stem_cell_transplantation,23,stem_cell_transplantation,CDE_ID:3090688,2406,0,...,2406,0,0,0,0,0,0,0,0,0
4,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,stem_cell_transplantation_type,24,stem_cell_transplantation_type,CDE_ID:2730901,2406,0,...,2406,0,0,0,0,0,0,0,0,0
5,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,clinical_trial_drug_classification,7,clinical_trail_drug_classification,CDE_ID:3378323,2406,3,...,2285,0,0,0,0,0,118,0,0,0
6,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,therapy_regimen_other,26,regimen_indication_notes,CDE_ID:2793516,2406,13,...,0,2393,0,0,0,0,0,0,0,0
7,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharma_type_other,17,therapy_type_notes,CDE_ID:2001762,2406,22,...,2384,0,0,0,0,0,0,0,0,0
8,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,death_days_to,12,days_to_death,CDE_ID:3165475,716,52,...,0,664,0,0,0,0,0,0,0,0
9,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,nte_er_positivity_other_scale,72,pos_finding_metastatic_breast_carcinoma_estrog...,CDE_ID:3131877,1097,0,...,1097,0,0,0,0,0,0,0,0,0


## Save likely high-value and likely weak review queues

These are audit-only review queues. They do not approve fields for cohort construction or endpoint selection.


In [8]:
likely_high_value_df = (
    audit_df.loc[
        (~audit_df['probable_field_group'].isin(['identifier / admin', 'other / unclear']))
        & (audit_df['missing_like_fraction'] <= 0.25)
        & (audit_df['distinct_non_missing_count'] >= 2)
    ]
    .sort_values(['table_name', 'missing_like_fraction', 'non_missing_count', 'source_position'], ascending=[True, True, False, True])
    .reset_index(drop=True)
)
likely_high_value_path = results_root / '23_clinical_core_likely_high_value_fields.tsv'
likely_high_value_df.to_csv(likely_high_value_path, sep='\t', index=False)

print(f'Saved: {likely_high_value_path}')
display(likely_high_value_df.head(25))


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\23_clinical_core_likely_high_value_fields.tsv


,audit_run_id,parse_run_id,source_run_id,table_name,field_name,source_position,alternate_column_name,cde_id_raw,row_count,non_missing_count,...,not_available_count,not_applicable_count,unknown_count,not_evaluated_count,discrepancy_count,na_count,n_a_count,null_count,none_count,nan_count
0,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_therapy_type,8,therapy_type,CDE_ID:2793530,2406,2399,...,7,0,0,0,0,0,0,0,0,0
1,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_tx_ongoing_indicator,10,therapy_ongoing,CDE_ID:3103479,2406,2391,...,15,0,0,0,0,0,0,0,0,0
2,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_therapy_drug_name,6,drug_name,CDE_ID:2975232,2406,2390,...,15,0,1,0,0,0,0,0,0,0
3,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_tx_started_days_to,9,days_to_drug_therapy_start,CDE_ID:3392465,2406,2288,...,116,0,0,0,2,0,0,0,0,0
4,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_tx_ended_days_to,11,days_to_drug_therapy_end,CDE_ID:3392470,2406,1805,...,601,0,0,0,0,0,0,0,0,0
5,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,vital_status,10,vital_status,CDE_ID:5,716,706,...,10,0,0,0,0,0,0,0,0,0
6,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,radiation_treatment_adjuvant,7,radiation_therapy,CDE_ID:2005312,716,692,...,11,0,12,0,1,0,0,0,0,0
7,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,followup_lost_to,6,lost_follow_up,CDE_ID:61333,716,690,...,26,0,0,0,0,0,0,0,0,0
8,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,pharmaceutical_tx_adjuvant,8,postoperative_rx_tx,CDE_ID:3397567,716,679,...,12,0,19,0,6,0,0,0,0,0
9,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,tumor_status,9,person_neoplasm_cancer_status,CDE_ID:2759550,716,677,...,18,0,21,0,0,0,0,0,0,0


In [9]:
likely_weak_df = (
    audit_df.loc[
        (~audit_df['probable_field_group'].isin(['identifier / admin']))
        & (
            (audit_df['missing_like_fraction'] >= 0.90)
            | (audit_df['non_missing_count'] == 0)
            | (audit_df['distinct_non_missing_count'] <= 1)
        )
    ]
    .sort_values(['table_name', 'missing_like_fraction', 'non_missing_count', 'source_position'], ascending=[True, False, True, True])
    .reset_index(drop=True)
)
likely_weak_path = results_root / '24_clinical_core_likely_weak_fields.tsv'
likely_weak_df.to_csv(likely_weak_path, sep='\t', index=False)

print(f'Saved: {likely_weak_path}')
display(likely_weak_df.head(25))


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\24_clinical_core_likely_weak_fields.tsv


,audit_run_id,parse_run_id,source_run_id,table_name,field_name,source_position,alternate_column_name,cde_id_raw,row_count,non_missing_count,...,not_available_count,not_applicable_count,unknown_count,not_evaluated_count,discrepancy_count,na_count,n_a_count,null_count,none_count,nan_count
0,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,days_to_stem_cell_transplantation,13,days_to_stem_cell_transplantation,CDE_ID:3414613,2406,0,...,2406,0,0,0,0,0,0,0,0,0
1,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharm_regimen,14,pharm_regimen,CDE_ID:3366758,2406,0,...,2406,0,0,0,0,0,0,0,0,0
2,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharm_regimen_other,15,pharm_regimen_other,CDE_ID:3366930,2406,0,...,2406,0,0,0,0,0,0,0,0,0
3,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,stem_cell_transplantation,23,stem_cell_transplantation,CDE_ID:3090688,2406,0,...,2406,0,0,0,0,0,0,0,0,0
4,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,stem_cell_transplantation_type,24,stem_cell_transplantation_type,CDE_ID:2730901,2406,0,...,2406,0,0,0,0,0,0,0,0,0
5,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,clinical_trial_drug_classification,7,clinical_trail_drug_classification,CDE_ID:3378323,2406,3,...,2285,0,0,0,0,0,118,0,0,0
6,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,therapy_regimen_other,26,regimen_indication_notes,CDE_ID:2793516,2406,13,...,0,2393,0,0,0,0,0,0,0,0
7,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharma_type_other,17,therapy_type_notes,CDE_ID:2001762,2406,22,...,2384,0,0,0,0,0,0,0,0,0
8,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,death_days_to,12,days_to_death,CDE_ID:3165475,716,52,...,0,664,0,0,0,0,0,0,0,0
9,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_patient,nte_er_positivity_other_scale,72,pos_finding_metastatic_breast_carcinoma_estrog...,CDE_ID:3131877,1097,0,...,1097,0,0,0,0,0,0,0,0,0
